Entity

In [1]:
from dataclasses import dataclass
from pathlib import Path
import os
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    checkpoint_dir: Path
    updated_model_path: Path    
    # hyperparams
    num_classes: int
    batch_size: int
    num_workers: int
    epochs: int
    seed: int
    lr_backbone: float
    lr_head: float
    weight_decay: float
    max_lr: float
    pct_start: float
    div_factor: float
    final_div_factor: float
    label_smoothing: float
    patience: int
    # augmentation
    use_mixup: bool
    use_cutmix: bool
    mixup_alpha: float
    cutmix_alpha: float
    mixup_cutmix_prob: float
    # ema
    use_ema: bool
    ema_decay: float

In [2]:
os.getcwd()

'd:\\Personal_projects\\Pyhton_proj\\AI-Food-Recognition-Nutrition-Assistant\\research'

In [3]:
os.chdir("..")

In [4]:
%pwd

'd:\\Personal_projects\\Pyhton_proj\\AI-Food-Recognition-Nutrition-Assistant'

Config Manager

In [5]:
from AI_Food_Recognition_Nutrition_Assistant.constants import *
from AI_Food_Recognition_Nutrition_Assistant.utils.common import read_yaml,create_directories

[2026-04-19 08:32:18,216: INFO: dsl_registry: Successfully registered DSL: cutedsl]
[2026-04-19 08:32:18,220: INFO: dsl_registry: Successfully registered DSL: triton]


In [6]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        config = self.config.training
        p = self.params
        create_directories([config.root_dir])
        create_directories([config.checkpoint_dir])
        data_ingestion_config = TrainingConfig(
            root_dir=Path(config.root_dir),
            trained_model_path=Path(config.trained_model_path),
            checkpoint_dir=Path(config.checkpoint_dir),
            updated_model_path=Path(self.config.prepare_base_model.updated_base_model_path),
            num_classes=p.training.num_classes,
            batch_size=p.training.batch_size,
            num_workers=p.training.num_workers,
            epochs=p.training.epochs,
            seed=p.training.seed,
            lr_backbone=p.training.lr_backbone,
            lr_head=p.training.lr_head,
            weight_decay=p.training.weight_decay,
            max_lr=p.training.max_lr,
            pct_start=p.training.pct_start,
            div_factor=p.training.div_factor,
            final_div_factor=p.training.final_div_factor,
            label_smoothing=p.training.label_smoothing,
            patience=p.training.patience,
            use_mixup=p.augmentation.use_mixup,
            use_cutmix=p.augmentation.use_cutmix,
            mixup_alpha=p.augmentation.mixup_alpha,
            cutmix_alpha=p.augmentation.cutmix_alpha,
            mixup_cutmix_prob=p.augmentation.mixup_cutmix_prob,
            use_ema=p.ema.use_ema,
            ema_decay=p.ema.ema_decay,
        )

        return data_ingestion_config

Training

In [7]:
import random
import copy
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader
from AI_Food_Recognition_Nutrition_Assistant.utils.common import set_seed, get_device, accuracy, precision_recall_f1
from AI_Food_Recognition_Nutrition_Assistant import logger

In [8]:
# ── EMA ──────────────────────────────────────────────────────────────────────
class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.999, device: str = "cpu"):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        self.ema.to(device)
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model: nn.Module) -> None:
        with torch.no_grad():
            msd = model.state_dict()
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v.copy_(v * self.decay + msd[k].detach() * (1.0 - self.decay))


# ── MixUp / CutMix ───────────────────────────────────────────────────────────
def _rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = (1.0 - lam) ** 0.5
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = random.randint(0, W), random.randint(0, H)
    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y2 = min(cy + cut_h // 2, H)
    return x1, y1, x2, y2


def mixup_data(x, y, alpha):
    lam = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def cutmix_data(x, y, alpha):
    lam = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    x1, y1, x2, y2 = _rand_bbox(x.size(), lam)
    x = x.clone()
    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (x.size(-1) * x.size(-2))
    return x, y, y[idx], lam


def mixed_loss(criterion, preds, y_a, y_b, lam):
    return lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)


# ── Trainer ───────────────────────────────────────────────────────────────────
class ModelTrainer:
    def __init__(self, config: TrainingConfig, model: nn.Module,
                 train_loader: DataLoader, val_loader: DataLoader):
        self.config = config
        self.device = get_device()
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader

        set_seed(config.seed)
        create_directories([config.root_dir])
        create_directories([config.checkpoint_dir])

        self.criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
        self.optimizer = AdamW([
            {"params": self.model.features.parameters(), "lr": config.lr_backbone},
            {"params": self.model.classifier.parameters(), "lr": config.lr_head},
        ], weight_decay=config.weight_decay)

        self.scheduler = OneCycleLR(
            self.optimizer,
            max_lr=config.max_lr,
            steps_per_epoch=len(train_loader),
            epochs=config.epochs,
            pct_start=config.pct_start,
            div_factor=config.div_factor,
            final_div_factor=config.final_div_factor,
        )
        self.scaler = torch.amp.GradScaler(device="cuda")
        self.ema = ModelEMA(self.model, config.ema_decay, self.device) if config.use_ema else None

    # ── augmentation dispatch ─────────────────────────────────────────────────
    def _apply_aug(self, x, y):
        if not (self.config.use_mixup or self.config.use_cutmix):
            return x, y, y, 1.0, False
        if random.random() >= self.config.mixup_cutmix_prob:
            return x, y, y, 1.0, False

        if self.config.use_mixup and self.config.use_cutmix:
            fn = mixup_data if random.random() < 0.5 else cutmix_data
        elif self.config.use_mixup:
            fn = mixup_data
        else:
            fn = cutmix_data

        alpha = self.config.mixup_alpha if fn is mixup_data else self.config.cutmix_alpha
        x, ya, yb, lam = fn(x, y, alpha)
        return x, ya, yb, lam, True

    # ── one epoch ─────────────────────────────────────────────────────────────
    def _train_epoch(self) -> float:
        self.model.train()
        total_loss = 0.0

        for imgs,labels in tqdm(self.train_loader, desc="Training", leave=False):
             imgs, labels = imgs.to(self.device), labels.to(self.device)
             imgs, ya, yb, lam, aug = self._apply_aug(imgs, labels)
             self.optimizer.zero_grad()
             with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                 out = self.model(imgs)
                 loss = mixed_loss(self.criterion, out, ya, yb, lam) if aug else self.criterion(out, labels)
             self.scaler.scale(loss).backward()
             self.scaler.step(self.optimizer)
             self.scaler.update()
             self.scheduler.step()
             if self.ema:
                 self.ema.update(self.model)
             total_loss += loss.item()
        return total_loss / len(self.train_loader)

        # for imgs, labels in self.train_loader:
        #     imgs, labels = imgs.to(self.device), labels.to(self.device)
        #     imgs, ya, yb, lam, aug = self._apply_aug(imgs, labels)
        #     self.optimizer.zero_grad()
        #     with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        #         out = self.model(imgs)
        #         loss = mixed_loss(self.criterion, out, ya, yb, lam) if aug else self.criterion(out, labels)
        #     self.scaler.scale(loss).backward()
        #     self.scaler.step(self.optimizer)
        #     self.scaler.update()
        #     self.scheduler.step()
        #     if self.ema:
        #         self.ema.update(self.model)
        #     total_loss += loss.item()
        # return total_loss / len(self.train_loader)

    def _validate(self):
        eval_model = self.ema.ema if (self.ema and self.config.use_ema) else self.model
        eval_model.eval()

        total_loss, c1, c5, n = 0.0, 0.0, 0.0, 0
        total_precision, total_recall, total_f1 = 0.0, 0.0, 0.0

        with torch.no_grad():
            for imgs, labels in self.val_loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)

                with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    out = eval_model(imgs)
                    loss = self.criterion(out, labels)

                total_loss += loss.item()

                # Accuracy
                (c1k, bs), (c5k, _) = accuracy(out, labels, topk=(1, 5))
                c1 += c1k.item()
                c5 += c5k.item()
                n += bs

                # Precision / Recall / F1
                p, r, f1 = precision_recall_f1(out, labels, self.config.num_classes)
                total_precision += p
                total_recall += r
                total_f1 += f1

        # Average over batches
        avg_precision = total_precision / len(self.val_loader)
        avg_recall = total_recall / len(self.val_loader)
        avg_f1 = total_f1 / len(self.val_loader)

        return (
            total_loss / len(self.val_loader),
            100.0 * c1 / n,
            100.0 * c5 / n,
            avg_precision,
            avg_recall,
            avg_f1,
        )


    # ── main train loop ───────────────────────────────────────────────────────
    def train(self) -> str:
        best_acc, wait = 0.0, 0
        best_path = self.config.checkpoint_dir / "best_model.pth"

        for epoch in range(self.config.epochs):
            train_loss = self._train_epoch()
            val_loss, top1, top5, avg_precision, avg_recall, avg_f1 = self._validate()
            logger.info(
                f"Epoch {epoch+1}/{self.config.epochs} | "
                f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                f"Top-1: {top1:.2f}% | Top-5: {top5:.2f}%"
                f" | Precision: {avg_precision:.4f} | Recall: {avg_recall:.4f} | F1: {avg_f1:.4f}"
            )
            if top1 > best_acc:
                best_acc = top1
                wait = 0
                torch.save(self.model.state_dict(), best_path)
                logger.info(f" New best Top-1: {best_acc:.2f}% — checkpoint saved.")
            else:
                wait += 1
                if wait >= self.config.patience:
                    logger.info("Early stopping triggered.")
                    break

        torch.save(self.model.state_dict(), self.config.trained_model_path)
        logger.info(f"Training complete. Best Top-1: {best_acc:.2f}%")
        return str(self.config.trained_model_path)

Pipeline

In [9]:
from AI_Food_Recognition_Nutrition_Assistant.pipeline.stage_02_data_preprocessing import DataPreprocessingPipeline
from AI_Food_Recognition_Nutrition_Assistant.pipeline.stage_03_prepare_base_model import PrepareBaseModelPipeline

In [10]:
try:
    config = ConfigurationManager()
    config = config.get_training_config()
    train_loader,val_loader,test_loader = DataPreprocessingPipeline().main()
    model = PrepareBaseModelPipeline().main()
    trainer = ModelTrainer(
        config=config,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
    )
    trainer.train()
except Exception as e:
    logger.exception(e)
    raise e

[2026-04-19 08:32:19,276: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-19 08:32:19,278: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-19 08:32:19,279: INFO: common: created directory at: artifacts]
[2026-04-19 08:32:19,279: INFO: common: created directory at: artifacts/training]
[2026-04-19 08:32:19,280: INFO: common: created directory at: artifacts/training/checkpoints]
[2026-04-19 08:32:19,282: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-19 08:32:19,283: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-19 08:32:19,284: INFO: common: created directory at: artifacts]
[2026-04-19 08:32:19,284: INFO: common: created directory at: artifacts/data_preprocessing]
[2026-04-19 08:32:

[2026-04-19 08:37:50,936: INFO: 3589805006: Epoch 1/60 | Train Loss: 2.8162 | Val Loss: 1.8863 | Top-1: 73.64% | Top-5: 91.99% | Precision: 0.2009 | Recall: 0.2015 | F1: 0.1988]
[2026-04-19 08:37:51,049: INFO: 3589805006:  New best Top-1: 73.64% — checkpoint saved.]


[2026-04-19 08:43:16,139: INFO: 3589805006: Epoch 2/60 | Train Loss: 1.8325 | Val Loss: 1.3423 | Top-1: 83.60% | Top-5: 96.03% | Precision: 0.2281 | Recall: 0.2282 | F1: 0.2265]
[2026-04-19 08:43:16,254: INFO: 3589805006:  New best Top-1: 83.60% — checkpoint saved.]


[2026-04-19 08:48:38,809: INFO: 3589805006: Epoch 3/60 | Train Loss: 1.7198 | Val Loss: 1.2587 | Top-1: 85.49% | Top-5: 96.54% | Precision: 0.2338 | Recall: 0.2336 | F1: 0.2321]
[2026-04-19 08:48:38,929: INFO: 3589805006:  New best Top-1: 85.49% — checkpoint saved.]


[2026-04-19 08:54:02,709: INFO: 3589805006: Epoch 4/60 | Train Loss: 1.7337 | Val Loss: 1.2523 | Top-1: 85.28% | Top-5: 96.48% | Precision: 0.2329 | Recall: 0.2329 | F1: 0.2314]


...

[2026-04-19 09:10:11,833: INFO: 3589805006: Epoch 7/60 | Train Loss: 1.7035 | Val Loss: 1.2707 | Top-1: 84.87% | Top-5: 96.08% | Precision: 0.2316 | Recall: 0.2317 | F1: 0.2301]


[2026-04-19 09:15:33,102: INFO: 3589805006: Epoch 8/60 | Train Loss: 1.6114 | Val Loss: 1.2757 | Top-1: 84.73% | Top-5: 95.84% | Precision: 0.2315 | Recall: 0.2315 | F1: 0.2299]


[2026-04-19 09:21:01,727: INFO: 3589805006: Epoch 9/60 | Train Loss: 1.5517 | Val Loss: 1.2906 | Top-1: 84.60% | Top-5: 95.67% | Precision: 0.2315 | Recall: 0.2314 | F1: 0.2298]


[2026-04-19 09:26:35,086: INFO: 3589805006: Epoch 10/60 | Train Loss: 1.5252 | Val Loss: 1.2979 | Top-1: 84.37% | Top-5: 95.59% | Precision: 0.2304 | Recall: 0.2304 | F1: 0.2288]


[2026-04-19 09:32:03,432: INFO: 3589805006: Epoch 11/60 | Train Loss: 1.4754 | Val Loss: 1.3050 | Top-1: 84.33% | Top-5: 95.82% | Precision: 0.2302 | Recall: 0.2302 | F1: 0.2286]
[2026-04-19 09:32:03,432: INFO: 3589805006: Early stopping triggered.]
[2026-04-19 09:32:03,533: INFO: 3589805006: Training complete. Best Top-1: 85.49%]
